# Importing Libraries

In [ ]:
# Standard Libraries
import os
from pathlib import Path
from random import sample
import shutil
import random
import csv

# Data Manipulation and Processing
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import yaml

# Image Handling and Processing
from PIL import Image, ImageOps, ImageDraw
import pillow_heif
import cv2

# Visualization
import matplotlib.pyplot as plt
import IPython.display as ipd
from IPython.display import display, HTML, Markdown

# Machine Learning and Model Interaction
import torch
from ultralytics import YOLO

# API Interaction
import requests
from roboflow import Roboflow


# Exploring the data

In [ ]:
# Set the path to the Fantasy Realms Data directory
data_directory = "Fantasy Realms Data"

In [ ]:
def print_directory_structure(start_path, indent_level=0):
    # Iterate through all items in the directory
    for item in os.listdir(start_path):
        # Get the full path of the item
        item_path = os.path.join(start_path, item)
        # Print the item with indentation
        print(" " * indent_level + "|-- " + item)
        
        # If the item is a directory, recursively call the function
        if os.path.isdir(item_path):
            print_directory_structure(item_path, indent_level + 4)

In [ ]:
# Print the structure
print("Directory structure for '{}':".format(data_directory))
print_directory_structure(data_directory)

In [ ]:
def print_directory_structure(start_path, indent_level=0, max_depth=2):
    # Check if the current depth has reached the maximum depth
    if indent_level // 4 >= max_depth:
        return

    # Iterate through all items in the directory
    for item in os.listdir(start_path):
        item_path = os.path.join(start_path, item)
        
        # Print the item with indentation if within the depth limit
        print(" " * indent_level + "|-- " + item)
        
        # If the item is a directory and not at max depth, recurse
        if os.path.isdir(item_path):
            print_directory_structure(item_path, indent_level + 4, max_depth)


In [ ]:
# Print the structure up to the second-last layer
print("Directory structure for '{}':".format(data_directory))
print_directory_structure(data_directory)

In [ ]:
def list_subfolders(directory):
    subfolders = []
    
    # Walk through the directory
    for root, dirs, _ in os.walk(directory):
        # For each subfolder in dirs, append the full path to the subfolders list
        for subfolder in dirs:
            subfolders.append(os.path.join(root, subfolder))
    
    return subfolders

In [ ]:
# Get and print the list of all subfolders
all_subfolders = list_subfolders(data_directory)
print("List of all subfolders:")
for folder in all_subfolders:
    print(folder)

In [ ]:
def list_first_layer_subfolders(directory):
    # List all items in the directory and filter to include only subfolders
    subfolders = [f for f in os.listdir(directory) if os.path.isdir(os.path.join(directory, f))]
    
    # Count the number of subfolders
    subfolder_count = len(subfolders)
    
    return subfolders, subfolder_count

In [ ]:
# Get the first layer of subfolders and their count
first_layer_subfolders, count = list_first_layer_subfolders(data_directory)
print("First layer subfolders:", first_layer_subfolders)
print("Count of first layer subfolders:", count)

In [ ]:
# Register HEIF format with Pillow
pillow_heif.register_heif_opener()

def convert_heic_to_jpeg(directory):
    found_heic_files = 0  # Counter for HEIC files

    # Walk through the directory
    for root, _, files in os.walk(directory):
        for file in files:
            # Check if the file has a .HEIC extension
            if file.lower().endswith(".heic"):
                found_heic_files += 1  # Increment the counter
                heic_path = os.path.join(root, file)

                # Open the .HEIC image
                try:
                    image = Image.open(heic_path)
                    # Define the new .jpeg path
                    jpeg_path = os.path.splitext(heic_path)[0] + ".jpeg"
                    
                    # Save the image as .jpeg
                    image.save(jpeg_path, "JPEG")
                    
                    # Delete the original .HEIC file
                    os.remove(heic_path)
                    print(f"Converted and deleted: {heic_path}")
                except Exception as e:
                    print(f"Failed to convert {heic_path}: {e}")

    # Check if no HEIC files were found
    if found_heic_files == 0:
        print("No .HEIC images found to convert.")

# Convert all .HEIC images to .jpeg and delete the originals
convert_heic_to_jpeg(data_directory)


# Table Data

In [ ]:
# notes from professor :

    #instant segmentation
    #yolo v8 library : oriented bounding box
    #edge detection algorithm
    # open MM detection
    #huggingface
    #132.231.12.72:8080

In [ ]:
# !pip install roboflow

# from roboflow import Roboflow
# rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
# project = rf.workspace("houssem-ouerdiane-nmtbp").project("fantasy-realms")
# version = project.version(3)
# dataset = version.download("yolov5")              

In [ ]:
# Directory containing the Table Data folder
data_dir = 'Table Data'

In [ ]:
# Roboflow model information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '3'  # The version number of your deployed model

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'


In [ ]:
RESIZE_DIM = (640, 640)  # Resize dimensions used during model training

# Function to get 10 random image paths from the directory
def get_random_images(data_dir, num_images=10):
    subfolders = [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]
    random_images = []
    
    for _ in range(num_images):
        random_folder = random.choice(subfolders)
        folder_path = os.path.join(data_dir, random_folder)
        images = [f for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
        random_image = random.choice(images)
        random_images.append(os.path.join(folder_path, random_image))
    
    return random_images

# Function to apply predictions on a given image using the Roboflow model
def predict_image(image_path):
    # Open and resize the image
    image = Image.open(image_path).convert("RGB")
    image_resized = image.resize(RESIZE_DIM)
    
    # Save the resized image temporarily for uploading
    temp_image_path = "temp_image.jpg"
    image_resized.save(temp_image_path)
    
    # Send the image to Roboflow API for prediction
    with open(temp_image_path, "rb") as img:
        response = requests.post(
            BASE_URL,
            params={"api_key": API_KEY},
            files={"file": img}
        )
    
    # Parse the prediction results
    predictions = response.json()
    return predictions, image_resized

# Function to draw bounding boxes on the image
def draw_bounding_boxes(image, predictions):
    draw = ImageDraw.Draw(image)
    if 'predictions' in predictions and predictions['predictions']:
        for pred in predictions['predictions']:
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            confidence = pred['confidence']
            label = pred['class']
            
            # Calculate bounding box coordinates
            x_min = x - width / 2
            y_min = y - height / 2
            x_max = x + width / 2
            y_max = y + height / 2
            
            # Draw the bounding box and label
            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
            draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
    return image

# Main function to process 10 random images
def process_random_images(data_dir):
    random_images = get_random_images(data_dir)
    
    for image_path in random_images:
        predictions, resized_image = predict_image(image_path)
        image_with_boxes = draw_bounding_boxes(resized_image.copy(), predictions)
        
        # Display the image with bounding boxes
        plt.figure(figsize=(8, 8))
        plt.imshow(image_with_boxes)
        plt.axis('off')
        plt.title("Predictions with Bounding Boxes")
        plt.show()


# Apply predictions on 10 random images
process_random_images(data_dir)


In [ ]:
# Function to get a random image from the Table Data folder
def get_random_image(data_dir):
    subfolders = [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]
    random_folder = random.choice(subfolders)
    folder_path = os.path.join(data_dir, random_folder)
    images = [f for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
    random_image = random.choice(images)
    return os.path.join(folder_path, random_image)

# Data augmentation techniques
def apply_augmentations(image):
    augmentations = {
        "Original": image,
        "Flip Horizontal": ImageOps.mirror(image),
        "Flip Vertical": ImageOps.flip(image),
        "Rotation -45°": image.rotate(-45, expand=True),
        "Rotation +45°": image.rotate(45, expand=True),
        "Grayscale": ImageOps.grayscale(image)
    }
    return augmentations

# Function to visualize the augmentations
def visualize_augmentations(augmentations):
    plt.figure(figsize=(15, 10))
    for i, (title, img) in enumerate(augmentations.items()):
        plt.subplot(2, 3, i + 1)
        plt.imshow(img, cmap="gray" if title == "Grayscale" else None)
        plt.title(title)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# Main function to process the random image
def process_random_image(data_dir):
    random_image_path = get_random_image(data_dir)
    original_image = Image.open(random_image_path).convert("RGB")
    
    # Apply augmentations
    augmentations = apply_augmentations(original_image)
    
    # Visualize the augmentations
    visualize_augmentations(augmentations)

# Process a random image and visualize augmentations
process_random_image(data_dir)


In [ ]:
# Define the model, specifying the pre-trained weights you want to use (e.g., yolov8n.pt)
model = YOLO("yolov8n.pt")  # Use YOLOv8 small model (yolov8n.pt)

# Train the model using the dataset specified in the data.yaml file
#model.train(data="data.yaml", epochs=150, batch=16, imgsz=640) #this step is only done once

In [ ]:
# the model take a lot of time to train (more than 6 hours), the results are captured into runs/train6
# there is also a pdf containing all the python output while training

In [ ]:
# Define the path to the data.yaml file
yaml_path = 'Fantasy-Realms-3/data.yaml' #table data

# Open and load the YAML file
with open(yaml_path, 'r') as file:
    data = yaml.safe_load(file)

# Print the contents of the YAML file
print(yaml.dump(data, sort_keys=False, default_flow_style=False))

In [ ]:
# Load the results.csv file
results_path = 'runs/detect/train6/results.csv' #train6 is for table data
df = pd.read_csv(results_path)

In [ ]:
# Display the table in a scrollable div
scrollable_table = df.to_html(index=False)
display(HTML(f"""
<div style="overflow:auto; height:400px; border:1px solid #ccc; padding:10px;">
    {scrollable_table}
</div>
"""))

In [ ]:
# Select the last epoch (max epoch)
last_epoch_data = df.loc[df['epoch'].idxmax()]

# Print each metric with its value
print("Metrics for the last epoch (Epoch {}):".format(last_epoch_data['epoch']))
for column in last_epoch_data.index:
    print(f"{column}: {last_epoch_data[column]}")

In [ ]:
# Extract epochs and precision values
epochs = df['epoch']
precision_b = df['metrics/precision(B)']

# Plot the evolution of precision over epochs
plt.figure(figsize=(10, 6))
plt.plot(epochs, precision_b, marker='o', linestyle='-', color='b')
plt.title('Evolution of Precision (B) by Epoch')
plt.xlabel('Epoch')
plt.ylabel('Precision (B)')
plt.grid(True)

# Set x-ticks at every 10 epochs
plt.xticks(ticks=epochs[::10], labels=epochs[::10], rotation=45)  

plt.tight_layout()
plt.show()

In [ ]:
# Plot 'metrics/recall(B)' by epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['metrics/recall(B)'], label='Recall (B)', color='red')
plt.xlabel('Epoch')
plt.ylabel('Recall (B)')
plt.title('Recall (B) by Epoch')
plt.xticks(df['epoch'][::10])  # Show every 10th epoch
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Plot 'metrics/mAP50(B)' by epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50 (B)', color='green')
plt.xlabel('Epoch')
plt.ylabel('mAP50 (B)')
plt.title('mAP50 (B) by Epoch')
plt.xticks(df['epoch'][::10])  # Show every 10th epoch
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Plot 'metrics/mAP50-95(B)' by epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95 (B)', color='blue')
plt.xlabel('Epoch')
plt.ylabel('mAP50-95 (B)')
plt.title('mAP50-95 (B) by Epoch')
plt.xticks(df['epoch'][::10])  # Show every 10th epoch
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Paths
train6_folder="runs/detect/train6"

# List of image file names (without extensions)
image_files = [
    'confusion_matrix',
    'confusion_matrix_normalized',
    'F1_curve',
    'labels',
    'labels_correlogram',
    'P_curve',
    'PR_curve',
    'R_curve',
    'results'
]

# Check for each image file and display it if it exists
for img_name in image_files:
    for ext in ['.png', '.jpg']:
        img_path = os.path.join(train6_folder, img_name + ext)
        if os.path.exists(img_path):
            image = Image.open(img_path)
            # Resize image and display
            resized_image = image.resize((1200, 800))  # Resize for better visibility
            ipd.display(resized_image)
            break

In [ ]:
# check this for metrics meanining : https://docs.ultralytics.com/guides/yolo-performance-metrics/#introduction

# Discard Data (Method 1)

In [ ]:
# Define the base folder path and the new "Discard Data" folder
base_directory = 'Fantasy Realms Data'
discard_data_directory = 'Discard Data'

In [ ]:
# Initialize a dictionary to store the count of images in each Discard folder
discard_counts = {}
total_images = 0  # Variable to store the total count of images

# Traverse the directory structure
for category in os.listdir(base_directory):
    category_path = os.path.join(base_directory, category)
    if os.path.isdir(category_path):
        for card in os.listdir(category_path):
            card_path = os.path.join(category_path, card)
            discard_folder_path = os.path.join(card_path, "Discard")
            
            # Check if the Discard folder exists
            if os.path.isdir(discard_folder_path):
                # Count the number of images (png, jpg)
                image_files = [f for f in os.listdir(discard_folder_path) if f.lower().endswith(('png', 'jpg', 'jpeg'))]
                image_count = len(image_files)
                discard_counts[os.path.join(category, card)] = image_count
                total_images += image_count  # Add to the total count

# Print the results
for card, count in discard_counts.items():
    print(f"{card}: {count} images")

# Print the total number of images
print(f"\nTotal number of images in all Discard folders: {total_images}")


In [ ]:
# # Create the discard data directory if it doesn't exist
# if not os.path.exists(discard_data_directory):
#     os.makedirs(discard_data_directory)

# # Initialize the total count and the target number of images (20% of 676)
# total_images = 676
# target_images = int(total_images * 0.20)  # 20% of 676
# images_selected = 0  # Counter for the images copied

# # Loop through each category folder (Army, Artifact, etc.)
# for category in os.listdir(base_folder):
#     category_path = os.path.join(base_folder, category)
    
#     # Skip if it's not a directory
#     if not os.path.isdir(category_path):
#         continue
    
#     print(f"Checking category: {category}")  # Debugging statement
    
#     # Loop through each card folder inside the category
#     for card in os.listdir(category_path):
#         card_path = os.path.join(category_path, card)
        
#         # Skip if it's not a directory
#         if not os.path.isdir(card_path):
#             continue
        
#         print(f"Checking card: {card}")  # Debugging statement
        
#         # Get all elements in the 'Discard' folder (we assume they are considered as images)
#         discard_folder_path = os.path.join(card_path, 'Discard')
#         if os.path.exists(discard_folder_path):
#             discard_files = os.listdir(discard_folder_path)
#             print(f"Found {len(discard_files)} items in 'Discard' folder of card '{card}'")  # Debugging statement
            
#             # Ensure at least 2 items are taken if there are more than 2 elements
#             if len(discard_files) >= 2:
#                 selected_images = random.sample(discard_files, min(2, len(discard_files)))
#                 images_selected += len(selected_images)

#                 # Copy selected items to the discard data folder
#                 for img_file in selected_images:
#                     original_image_path = os.path.join(discard_folder_path, img_file)
#                     shutil.copy(original_image_path, discard_data_directory)
#                     print(f"Copied {img_file} to Discard Data")  # Debugging statement
        
#         # Stop if the target number of images has been reached
#         if images_selected >= target_images:
#             break
    
#     # Stop if the target number of images has been reached
#     if images_selected >= target_images:
#         break

# # Verify the count of images in the Discard Data folder
# discard_data_images = os.listdir(discard_data_directory)
# print(f"Total images copied to 'Discard Data': {len(discard_data_images)}")



In [ ]:
# we used the previous code to get about 20% of the discard data that we have to put it into labeling (it was ensured that 2 images of each cards were taken randomy)

In [ ]:
# notes from the professor :
# the prediction system and metrics for the one card model is good (we can suppose it is done)
# for the prediction of multiple cards on the table (discard deck), in addition to using the annotation methods, the professor urged to try another method :
# --> the method is to try using the bounding boxes generated by the algorithm to extract each card on its own and the use the one card prediction at a time to predict each card on its own
# for the prediction of the cards on the hand, the professor urged the use of OCR such as pytessearact in order to predict the card using the card category and the card power

In [ ]:
# rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
# project = rf.workspace("houssem-ouerdiane-nmtbp").project("fantasy-realms")
# version = project.version(4)
# dataset = version.download("yolov5")

In [ ]:
#the previous code was used to download the dataset from Roboflow

In [ ]:
# Directory containing the "Discard Data" folder
data_dir = 'Discard Data'

In [ ]:
# Roboflow model information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '4'  # The version number of your deployed model

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'

In [ ]:
# Roboflow model information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '4'  # The version number of your deployed model

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'

RESIZE_DIM = (640, 640)  # Resize dimensions used during model training

# Function to get 10 random image paths from the "Discard Data" folder
def get_random_images(data_dir, num_images=10):
    images = [f for f in os.listdir(data_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random_images = random.sample(images, min(num_images, len(images)))
    return [os.path.join(data_dir, img) for img in random_images]

# Function to apply predictions on a given image using the Roboflow model
def predict_image(image_path):
    # Open and resize the image
    image = Image.open(image_path).convert("RGB")
    image_resized = image.resize(RESIZE_DIM)
    
    # Save the resized image temporarily for uploading
    temp_image_path = "temp_image.jpg"
    image_resized.save(temp_image_path)
    
    # Send the image to Roboflow API for prediction
    with open(temp_image_path, "rb") as img:
        response = requests.post(
            BASE_URL,
            params={"api_key": API_KEY},
            files={"file": img}
        )
    
    # Parse the prediction results
    predictions = response.json()
    return predictions, image_resized

# Function to draw bounding boxes on the image
def draw_bounding_boxes(image, predictions):
    draw = ImageDraw.Draw(image)
    if 'predictions' in predictions and predictions['predictions']:
        for pred in predictions['predictions']:
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            confidence = pred['confidence']
            label = pred['class']
            
            # Calculate bounding box coordinates
            x_min = x - width / 2
            y_min = y - height / 2
            x_max = x + width / 2
            y_max = y + height / 2
            
            # Draw the bounding box and label
            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
            draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
    return image

# Main function to process 10 random images
def process_random_images(data_dir):
    random_images = get_random_images(data_dir)
    
    for image_path in random_images:
        predictions, resized_image = predict_image(image_path)
        image_with_boxes = draw_bounding_boxes(resized_image.copy(), predictions)
        
        # Display the image with bounding boxes
        plt.figure(figsize=(8, 8))
        plt.imshow(image_with_boxes)
        plt.axis('off')
        plt.title("Predictions with Bounding Boxes")
        plt.show()

# Apply predictions on 10 random images
process_random_images(data_dir)


In [ ]:
# Define the path to the data.yaml file
yaml_path = 'Fantasy-Realms-4/data.yaml'

# Open and load the YAML file
with open(yaml_path, 'r') as file:
    data = yaml.safe_load(file)

# Print the contents of the YAML file
print(yaml.dump(data, sort_keys=False, default_flow_style=False))

In [ ]:
# Define the model, specifying the pre-trained weights you want to use (e.g., yolov8n.pt)
model = YOLO("yolov8n.pt")  # Use YOLOv8 small model (yolov8n.pt)

#Train the model using the dataset specified in the data.yaml file
#model.train(data="data_discard.yaml", epochs=150, batch=16, imgsz=640)
# already trained (folder train)

In [ ]:
# Load the results.csv file
results_path = 'runs/detect/train/results.csv' #train is for discard data
df = pd.read_csv(results_path)

In [ ]:
# Display the table in a scrollable div
scrollable_table = df.to_html(index=False)
display(HTML(f"""
<div style="overflow:auto; height:400px; border:1px solid #ccc; padding:10px;">
    {scrollable_table}
</div>
"""))

In [ ]:
# Select the last epoch (max epoch)
last_epoch_data = df.loc[df['epoch'].idxmax()]

# Print each metric with its value
print("Metrics for the last epoch (Epoch {}):".format(last_epoch_data['epoch']))
for column in last_epoch_data.index:
    print(f"{column}: {last_epoch_data[column]}")

In [ ]:
# Extract epochs and precision values
epochs = df['epoch']
precision_b = df['metrics/precision(B)']

# Plot the evolution of precision over epochs
plt.figure(figsize=(10, 6))
plt.plot(epochs, precision_b, marker='o', linestyle='-', color='b')
plt.title('Evolution of Precision (B) by Epoch')
plt.xlabel('Epoch')
plt.ylabel('Precision (B)')
plt.grid(True)

# Set x-ticks at every 10 epochs
plt.xticks(ticks=epochs[::10], labels=epochs[::10], rotation=45)  

plt.tight_layout()
plt.show()

In [ ]:
# Plot 'metrics/recall(B)' by epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['metrics/recall(B)'], label='Recall (B)', color='red')
plt.xlabel('Epoch')
plt.ylabel('Recall (B)')
plt.title('Recall (B) by Epoch')
plt.xticks(df['epoch'][::10])  # Show every 10th epoch
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Plot 'metrics/mAP50(B)' by epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50 (B)', color='green')
plt.xlabel('Epoch')
plt.ylabel('mAP50 (B)')
plt.title('mAP50 (B) by Epoch')
plt.xticks(df['epoch'][::10])  # Show every 10th epoch
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Plot 'metrics/mAP50-95(B)' by epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95 (B)', color='blue')
plt.xlabel('Epoch')
plt.ylabel('mAP50-95 (B)')
plt.title('mAP50-95 (B) by Epoch')
plt.xticks(df['epoch'][::10])  # Show every 10th epoch
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Paths
train_folder="runs/detect/train"

# List of image file names (without extensions)
image_files = [
    'confusion_matrix',
    'confusion_matrix_normalized',
    'F1_curve',
    'labels',
    'labels_correlogram',
    'P_curve',
    'PR_curve',
    'R_curve',
    'results'
]

# Check for each image file and display it if it exists
for img_name in image_files:
    for ext in ['.png', '.jpg']:
        img_path = os.path.join(train_folder, img_name + ext)
        if os.path.exists(img_path):
            image = Image.open(img_path)
            # Resize image and display
            resized_image = image.resize((1200, 800))  # Resize for better visibility
            ipd.display(resized_image)
            break

# Discard Data (Method 2)

In [ ]:
# Roboflow model information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '4'  # The version number of your deployed model

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'
RESIZE_DIM = (640, 640)  # Resize dimensions used during model training

# Function to get one random image path from the "Discard Data" folder
def get_random_image(data_dir):
    images = [f for f in os.listdir(data_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random_image = random.choice(images)
    return os.path.join(data_dir, random_image)

# Function to apply predictions on a given image using the Roboflow model
def predict_image(image_path):
    # Open the image
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    
    # Resize the image for Roboflow
    image_resized = image.resize(RESIZE_DIM)
    temp_image_path = "temp_image.jpg"
    image_resized.save(temp_image_path)
    
    # Send the image to Roboflow API for prediction
    with open(temp_image_path, "rb") as img:
        response = requests.post(
            BASE_URL,
            params={"api_key": API_KEY},
            files={"file": img}
        )
    
    # Parse the prediction results
    predictions = response.json()
    return predictions, image, width, height

# Function to draw bounding boxes on the image for visualization
def draw_bounding_boxes(image, predictions, orig_width, orig_height):
    draw = ImageDraw.Draw(image)
    if 'predictions' in predictions and predictions['predictions']:
        for pred in predictions['predictions']:
            # Normalize bounding box coordinates back to the original image dimensions
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            x_min = int((x - width / 2) * orig_width / RESIZE_DIM[0])
            y_min = int((y - height / 2) * orig_height / RESIZE_DIM[1])
            x_max = int((x + width / 2) * orig_width / RESIZE_DIM[0])
            y_max = int((y + height / 2) * orig_height / RESIZE_DIM[1])
            
            # Draw the bounding box and label
            label = pred['class']
            confidence = pred['confidence']
            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
            draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
    return image

# Function to extract cards using bounding boxes
def extract_cards(image, predictions, output_folder, image_name):
    # Create a subfolder with the name of the image (without extension)
    image_subfolder = os.path.join(output_folder, os.path.splitext(image_name)[0])
    os.makedirs(image_subfolder, exist_ok=True)

    if 'predictions' in predictions and predictions['predictions']:
        for i, pred in enumerate(predictions['predictions']):
            # Normalize bounding box coordinates back to the original image dimensions
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            x_min = int((x - width / 2) * image.width / RESIZE_DIM[0])
            y_min = int((y - height / 2) * image.height / RESIZE_DIM[1])
            x_max = int((x + width / 2) * image.width / RESIZE_DIM[0])
            y_max = int((y + height / 2) * image.height / RESIZE_DIM[1])
            
            # Crop the card from the image
            card = image.crop((x_min, y_min, x_max, y_max))
            
            # Save the cropped card in the subfolder
            card.save(os.path.join(image_subfolder, f"card_{i + 1}.jpg"))
        print(f"Extracted {len(predictions['predictions'])} cards to {image_subfolder}.")
    else:
        print("No predictions found.")

# Main function to process one random image
def process_random_image(data_dir, output_folder):
    random_image_path = get_random_image(data_dir)
    image_name = os.path.basename(random_image_path)
    print(f"Processing image: {random_image_path}")
    
    # Predict and visualize predictions
    predictions, original_image, orig_width, orig_height = predict_image(random_image_path)
    image_with_boxes = draw_bounding_boxes(original_image.copy(), predictions, orig_width, orig_height)
    
    # Display the original image with bounding boxes
    plt.figure(figsize=(8, 8))
    plt.imshow(image_with_boxes)
    plt.axis('off')
    plt.title("Extracted Cards")
    plt.show()
    
    # Extract cards after visualization
    extract_cards(original_image, predictions, output_folder, image_name)

# Folder paths
data_dir = "Discard Data"
output_folder = "Extracted Cards"
os.makedirs(output_folder, exist_ok=True)

# Process one random image
process_random_image(data_dir, output_folder)



In [ ]:
# Roboflow API Information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '3'  # The version number of your deployed model
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'

In [ ]:
# Roboflow API Information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '3'  # The version number of your deployed model
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'
RESIZE_DIM = (640, 640)  # Resize dimensions used during model training

# Function to get the path of the last created subfolder
def get_last_subfolder(base_dir):
    subfolders = [os.path.join(base_dir, folder) for folder in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, folder))]
    if not subfolders:
        raise ValueError("No subfolders found in the specified directory.")
    return max(subfolders, key=os.path.getmtime)  # Get the most recently modified folder

# Function to apply predictions on a given image using the Roboflow model
def predict_image(image_path):
    # Open and resize the image
    image = Image.open(image_path).convert("RGB")
    image_resized = image.resize(RESIZE_DIM)
    
    # Save the resized image temporarily for uploading
    temp_image_path = "temp_image.jpg"
    image_resized.save(temp_image_path)
    
    # Send the image to Roboflow API for prediction
    with open(temp_image_path, "rb") as img:
        response = requests.post(
            BASE_URL,
            params={"api_key": API_KEY},
            files={"file": img}
        )
    
    # Parse the prediction results
    predictions = response.json()
    return predictions, image_resized

# Function to draw bounding boxes and labels on the image
def draw_bounding_boxes(image, predictions):
    draw = ImageDraw.Draw(image)
    if 'predictions' in predictions and predictions['predictions']:
        for pred in predictions['predictions']:
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            confidence = pred['confidence']
            label = pred['class']
            
            # Calculate bounding box coordinates
            x_min = x - width / 2
            y_min = y - height / 2
            x_max = x + width / 2
            y_max = y + height / 2
            
            # Draw the bounding box and label with confidence
            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=3)
            draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
    return image

# Function to predict and display results for all images in a folder
def predict_cards_in_folder(folder_path):
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if not image_files:
        print("No images found in the folder.")
        return

    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)
        print(f"Predicting for image: {image_path}")
        
        # Get predictions
        predictions, resized_image = predict_image(image_path)
        
        # Draw bounding boxes with predictions
        image_with_boxes = draw_bounding_boxes(resized_image.copy(), predictions)
        
        # Display the image with bounding boxes
        plt.figure(figsize=(8, 8))
        plt.imshow(image_with_boxes)
        plt.axis('off')
        plt.title(f"Predictions for {image_file}")
        plt.show()
        
        # Print predictions
        if 'predictions' in predictions and predictions['predictions']:
            for pred in predictions['predictions']:
                label = pred['class']
                confidence = pred['confidence']
                print(f"Predicted: {label} with confidence: {confidence:.2f}")
        else:
            print("No predictions found for this image.")

# Main process
extracted_cards_dir = "Extracted Cards"  # Path to the folder containing extracted cards
last_subfolder = get_last_subfolder(extracted_cards_dir)
print(f"Processing the last created subfolder: {last_subfolder}")

# Predict cards in the last subfolder
predict_cards_in_folder(last_subfolder)



# Hand Data (Method 1)

In [ ]:
# # Define paths
# base_folder = "Fantasy Realms Data"  # Path to your main dataset
# output_folder = "Hand Data"         # Folder to store extracted images

# # Create the output folder if it doesn't exist
# if not os.path.exists(output_folder):
#     os.makedirs(output_folder)

# # Iterate through each category folder
# for category in os.listdir(base_folder):
#     category_path = os.path.join(base_folder, category)
    
#     if not os.path.isdir(category_path):
#         continue  # Skip if not a directory
    
#     print(f"Processing category: {category}")
    
#     # Iterate through each card folder in the category
#     for card in os.listdir(category_path):
#         card_path = os.path.join(category_path, card, "Hand")  # Specify "Hand" subfolder
#         if not os.path.isdir(card_path):
#             continue  # Skip if the "Hand" folder doesn't exist
        
#         print(f"Processing card: {card}")
        
#         # Get the list of all files in the "Hand" folder
#         img_files = os.listdir(card_path)
#         if len(img_files) == 0:
#             print(f"No images found in card: {card}")
#             continue  # Skip if the folder is empty
        
#         # Randomly select 2 images, or take all if less than 2
#         selected_files = sample(img_files, min(2, len(img_files)))
        
#         # Copy the selected images to the output folder
#         for file in selected_files:
#             src_path = os.path.join(card_path, file)
#             dest_path = os.path.join(output_folder, f"{card}_{file}")
#             shutil.copy(src_path, dest_path)
#             print(f"Copied: {src_path} -> {dest_path}")

# # Count the total number of images in the "Hand Data" folder
# total_images = len(os.listdir(output_folder))
# print(f"Total images copied to '{output_folder}': {total_images}")


In [ ]:
#the previous code was used to create the Hand data extraction (20%)

In [ ]:
def count_images_in_folder(folder_path):
    # Supported image file extensions
    supported_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.gif')
    image_count = 0

    # Walk through the directory and count image files
    for root, dirs, files in os.walk(folder_path):
        image_count += sum(1 for file in files if file.lower().endswith(supported_extensions))

    return image_count

# Example usage
hand_data_folder = "Hand Data"  # Replace with the actual path to your Hand Data folder
total_images = count_images_in_folder(hand_data_folder)
print(f"Total number of images in the Hand Data folder: {total_images}")

In [ ]:
#rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
#project = rf.workspace("houssem-ouerdiane-nmtbp").project("fantasy-realms")
#version = project.version(5)
#dataset = version.download("yolov5")

In [ ]:
# the precious code was used to download the dataset from Roboflow

In [ ]:
# Roboflow model information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '5'  # The version number of your deployed model

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'

In [ ]:
# Directory containing the "Hand Data" folder
data_dir = 'Hand Data'

In [ ]:
# Roboflow model information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '5'  # The version number of your deployed model

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'

RESIZE_DIM = (640, 640)  # Resize dimensions used during model training

# Function to get 10 random image paths from the "Discard Data" folder
def get_random_images(data_dir, num_images=10):
    images = [f for f in os.listdir(data_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random_images = random.sample(images, min(num_images, len(images)))
    return [os.path.join(data_dir, img) for img in random_images]

# Function to apply predictions on a given image using the Roboflow model
def predict_image(image_path):
    # Open and resize the image
    image = Image.open(image_path).convert("RGB")
    image_resized = image.resize(RESIZE_DIM)
    
    # Save the resized image temporarily for uploading
    temp_image_path = "temp_image.jpg"
    image_resized.save(temp_image_path)
    
    # Send the image to Roboflow API for prediction
    with open(temp_image_path, "rb") as img:
        response = requests.post(
            BASE_URL,
            params={"api_key": API_KEY},
            files={"file": img}
        )
    
    # Parse the prediction results
    predictions = response.json()
    return predictions, image_resized

# Function to draw bounding boxes on the image
def draw_bounding_boxes(image, predictions):
    draw = ImageDraw.Draw(image)
    prediction_text = []

    if 'predictions' in predictions and predictions['predictions']:
        for pred in predictions['predictions']:
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            confidence = pred['confidence']
            label = pred['class']
            
            # Calculate bounding box coordinates
            x_min = x - width / 2
            y_min = y - height / 2
            x_max = x + width / 2
            y_max = y + height / 2
            
            # Draw the bounding box and label
            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
            draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
            
            # Append prediction details for display
            prediction_text.append(f"{label}: {confidence:.2f}")
    return image, prediction_text

# Main function to process 10 random images
def process_random_images(data_dir):
    random_images = get_random_images(data_dir)
    
    for image_path in random_images:
        predictions, resized_image = predict_image(image_path)
        image_with_boxes, prediction_text = draw_bounding_boxes(resized_image.copy(), predictions)
        
        # Display the image with bounding boxes
        plt.figure(figsize=(8, 8))
        plt.imshow(image_with_boxes)
        plt.axis('off')
        plt.title("Predictions with Bounding Boxes")
        plt.show()
        
        # Display predictions below the image
        display(Markdown("### Predictions:"))
        for text in prediction_text:
            display(Markdown(f"- {text}"))

# Example usage
process_random_images(data_dir)


In [ ]:
# Define the path to the data.yaml file
yaml_path = 'Fantasy-Realms-5/data.yaml'

# Open and load the YAML file
with open(yaml_path, 'r') as file:
    data = yaml.safe_load(file)

# Print the contents of the YAML file
print(yaml.dump(data, sort_keys=False, default_flow_style=False))

In [ ]:
# Define the model, specifying the pre-trained weights you want to use (e.g., yolov8n.pt)
model = YOLO("yolov8n.pt")  # Use YOLOv8 small model (yolov8n.pt)

#Train the model using the dataset specified in the data.yaml file
#model.train(data="data_hand.yaml", epochs=150, batch=16, imgsz=640)
#already trained

In [ ]:
#saved in train9

In [ ]:
# Load the results.csv file
results_path = 'runs/detect/train9/results.csv' #train is for hand data
df = pd.read_csv(results_path)

In [ ]:
# Display the table in a scrollable div
scrollable_table = df.to_html(index=False)
display(HTML(f"""
<div style="overflow:auto; height:400px; border:1px solid #ccc; padding:10px;">
    {scrollable_table}
</div>
"""))

In [ ]:
# Select the last epoch (max epoch)
last_epoch_data = df.loc[df['epoch'].idxmax()]

# Print each metric with its value
print("Metrics for the last epoch (Epoch {}):".format(last_epoch_data['epoch']))
for column in last_epoch_data.index:
    print(f"{column}: {last_epoch_data[column]}")

In [ ]:
# Extract epochs and precision values
epochs = df['epoch']
precision_b = df['metrics/precision(B)']

# Plot the evolution of precision over epochs
plt.figure(figsize=(10, 6))
plt.plot(epochs, precision_b, marker='o', linestyle='-', color='b')
plt.title('Evolution of Precision (B) by Epoch')
plt.xlabel('Epoch')
plt.ylabel('Precision (B)')
plt.grid(True)

# Set x-ticks at every 10 epochs
plt.xticks(ticks=epochs[::10], labels=epochs[::10], rotation=45)  

plt.tight_layout()
plt.show()

In [ ]:
# Plot 'metrics/recall(B)' by epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['metrics/recall(B)'], label='Recall (B)', color='red')
plt.xlabel('Epoch')
plt.ylabel('Recall (B)')
plt.title('Recall (B) by Epoch')
plt.xticks(df['epoch'][::10])  # Show every 10th epoch
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Plot 'metrics/mAP50(B)' by epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50 (B)', color='green')
plt.xlabel('Epoch')
plt.ylabel('mAP50 (B)')
plt.title('mAP50 (B) by Epoch')
plt.xticks(df['epoch'][::10])  # Show every 10th epoch
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Plot 'metrics/mAP50-95(B)' by epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95 (B)', color='blue')
plt.xlabel('Epoch')
plt.ylabel('mAP50-95 (B)')
plt.title('mAP50-95 (B) by Epoch')
plt.xticks(df['epoch'][::10])  # Show every 10th epoch
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Paths
train_folder="runs/detect/train9"

# List of image file names (without extensions)
image_files = [
    'confusion_matrix',
    'confusion_matrix_normalized',
    'F1_curve',
    'labels',
    'labels_correlogram',
    'P_curve',
    'PR_curve',
    'R_curve',
    'results'
]

# Check for each image file and display it if it exists
for img_name in image_files:
    for ext in ['.png', '.jpg']:
        img_path = os.path.join(train_folder, img_name + ext)
        if os.path.exists(img_path):
            image = Image.open(img_path)
            # Resize image and display
            resized_image = image.resize((1200, 800))  # Resize for better visibility
            ipd.display(resized_image)
            break

# Hand Data (Method 2)

In [ ]:
# Roboflow model information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '5'  # The version number of your deployed model

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'
RESIZE_DIM = (640, 640)  # Resize dimensions used during model training

# Function to get one random image path from the "Discard Data" folder
def get_random_image(data_dir):
    images = [f for f in os.listdir(data_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random_image = random.choice(images)
    return os.path.join(data_dir, random_image)

# Function to apply predictions on a given image using the Roboflow model
def predict_image(image_path):
    # Open the image
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    
    # Resize the image for Roboflow
    image_resized = image.resize(RESIZE_DIM)
    temp_image_path = "temp_image.jpg"
    image_resized.save(temp_image_path)
    
    # Send the image to Roboflow API for prediction
    with open(temp_image_path, "rb") as img:
        response = requests.post(
            BASE_URL,
            params={"api_key": API_KEY},
            files={"file": img}
        )
    
    # Parse the prediction results
    predictions = response.json()
    return predictions, image, width, height

# Function to draw bounding boxes on the image for visualization
def draw_bounding_boxes(image, predictions, orig_width, orig_height):
    draw = ImageDraw.Draw(image)
    if 'predictions' in predictions and predictions['predictions']:
        for pred in predictions['predictions']:
            # Normalize bounding box coordinates back to the original image dimensions
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            x_min = int((x - width / 2) * orig_width / RESIZE_DIM[0])
            y_min = int((y - height / 2) * orig_height / RESIZE_DIM[1])
            x_max = int((x + width / 2) * orig_width / RESIZE_DIM[0])
            y_max = int((y + height / 2) * orig_height / RESIZE_DIM[1])
            
            # Draw the bounding box and label
            label = pred['class']
            confidence = pred['confidence']
            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
            draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
    return image

# Function to extract cards using bounding boxes
def extract_cards(image, predictions, output_folder, image_name):
    # Create a subfolder with the name of the image (without extension)
    image_subfolder = os.path.join(output_folder, os.path.splitext(image_name)[0])
    os.makedirs(image_subfolder, exist_ok=True)

    if 'predictions' in predictions and predictions['predictions']:
        for i, pred in enumerate(predictions['predictions']):
            # Normalize bounding box coordinates back to the original image dimensions
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            x_min = int((x - width / 2) * image.width / RESIZE_DIM[0])
            y_min = int((y - height / 2) * image.height / RESIZE_DIM[1])
            x_max = int((x + width / 2) * image.width / RESIZE_DIM[0])
            y_max = int((y + height / 2) * image.height / RESIZE_DIM[1])
            
            # Crop the card from the image
            card = image.crop((x_min, y_min, x_max, y_max))
            
            # Save the cropped card in the subfolder
            card.save(os.path.join(image_subfolder, f"card_{i + 1}.jpg"))
        print(f"Extracted {len(predictions['predictions'])} cards to {image_subfolder}.")
    else:
        print("No predictions found.")

# Main function to process one random image
def process_random_image(data_dir, output_folder):
    random_image_path = get_random_image(data_dir)
    image_name = os.path.basename(random_image_path)
    print(f"Processing image: {random_image_path}")
    
    # Predict and visualize predictions
    predictions, original_image, orig_width, orig_height = predict_image(random_image_path)
    image_with_boxes = draw_bounding_boxes(original_image.copy(), predictions, orig_width, orig_height)
    
    # Display the original image with bounding boxes
    plt.figure(figsize=(8, 8))
    plt.imshow(image_with_boxes)
    plt.axis('off')
    plt.title("Extracted Cards")
    plt.show()
    
    # Extract cards after visualization
    extract_cards(original_image, predictions, output_folder, image_name)

# Folder paths
data_dir = "Hand Data"
output_folder = "Extracted Cards (Hand)"
os.makedirs(output_folder, exist_ok=True)

# Process one random image
process_random_image(data_dir, output_folder)

In [ ]:
# Function to find the most recently created folder
def get_latest_folder(output_folder):
    subfolders = [
        os.path.join(output_folder, folder)
        for folder in os.listdir(output_folder)
        if os.path.isdir(os.path.join(output_folder, folder))
    ]
    if subfolders:
        latest_folder = max(subfolders, key=os.path.getmtime)
        return latest_folder
    return None

# Function to display images from the latest folder
def display_images_from_latest_folder(output_folder):
    latest_folder = get_latest_folder(output_folder)
    if latest_folder:
        print(f"Displaying images from the latest folder: {latest_folder}")
        for image_file in os.listdir(latest_folder):
            if image_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_path = os.path.join(latest_folder, image_file)
                image = Image.open(image_path)
                
                # Display the image
                plt.figure(figsize=(4, 4))
                plt.imshow(image)
                plt.axis('off')
                plt.title(f"Card: {image_file}")
                plt.show()
    else:
        print("No folders found in the output directory.")

# Display images from the most recently created folder
display_images_from_latest_folder(output_folder)



# Optimizing Hand Data

In [ ]:
# Roboflow model information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '5'  # The version number of your deployed model

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'

RESIZE_DIM = (640, 640)  # Resize dimensions used during model training
CONFIDENCE_THRESHOLD = 0.1  # Set confidence threshold to 0.1

# Function to get 10 random image paths from the "Discard Data" folder
def get_random_images(data_dir, num_images=10):
    images = [f for f in os.listdir(data_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random_images = random.sample(images, min(num_images, len(images)))
    return [os.path.join(data_dir, img) for img in random_images]

# Function to apply predictions on a given image using the Roboflow model
def predict_image(image_path):
    # Open and resize the image
    image = Image.open(image_path).convert("RGB")
    image_resized = image.resize(RESIZE_DIM)
    
    # Save the resized image temporarily for uploading
    temp_image_path = "temp_image.jpg"
    image_resized.save(temp_image_path)
    
    # Send the image to Roboflow API for prediction
    with open(temp_image_path, "rb") as img:
        response = requests.post(
            BASE_URL,
            params={
                "api_key": API_KEY,
                "confidence": CONFIDENCE_THRESHOLD  # Add confidence parameter here
            },
            files={"file": img}
        )
    
    # Parse the prediction results
    predictions = response.json()
    return predictions, image_resized

# Function to draw bounding boxes on the image
def draw_bounding_boxes(image, predictions):
    draw = ImageDraw.Draw(image)
    prediction_text = []

    if 'predictions' in predictions and predictions['predictions']:
        # Sort predictions by confidence in descending order
        sorted_predictions = sorted(predictions['predictions'], key=lambda x: x['confidence'], reverse=True)
        
        # Keep only the top 7 predictions
        top_predictions = sorted_predictions[:7]

        for pred in top_predictions:
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            confidence = pred['confidence']
            label = pred['class']
            
            # Calculate bounding box coordinates
            x_min = x - width / 2
            y_min = y - height / 2
            x_max = x + width / 2
            y_max = y + height / 2
            
            # Draw the bounding box and label
            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
            draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
            
            # Append prediction details for display
            prediction_text.append(f"{label}: {confidence:.2f}")
    return image, prediction_text

# Main function to process 10 random images
def process_random_images(data_dir):
    random_images = get_random_images(data_dir)
    
    for image_path in random_images:
        predictions, resized_image = predict_image(image_path)
        image_with_boxes, prediction_text = draw_bounding_boxes(resized_image.copy(), predictions)
        
        # Display the image with bounding boxes
        plt.figure(figsize=(8, 8))
        plt.imshow(image_with_boxes)
        plt.axis('off')
        plt.title("Predictions with Bounding Boxes")
        plt.show()
        
        # Display predictions below the image
        display(Markdown("### Predictions:"))
        for text in prediction_text:
            display(Markdown(f"- {text}"))

# Example usage
process_random_images(data_dir)


In [ ]:
# notes from professor
# change confidence rate for the hand data and pick the best 7 cards that has the best confidence
# try to test OCR on single cards
# do a manual evaluation with fresh data
# try to do segmentation for the hand data using https://github.com/facebookresearch/sam2
# do the recomendation systems using classes

# OCR testing (Single Cards)

In [ ]:
# import os
# import random
# from PIL import Image
# import matplotlib.pyplot as plt

# # Resize dimensions used during model training
# RESIZE_DIM = (640, 640)

# # Function to get 10 random card image paths from the directory
# def get_random_card_images(data_dir, num_images=10):
#     subfolders = [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]
#     random_images = []
    
#     for _ in range(num_images):
#         random_folder = random.choice(subfolders)  # Select a random subfolder
#         folder_path = os.path.join(data_dir, random_folder)
#         images = [f for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
        
#         if images:  # Ensure there are images in the folder
#             random_image = random.choice(images)
#             random_images.append(os.path.join(folder_path, random_image))
    
#     return random_images

# # Directory containing the Table Data folder
# data_dir = 'Table Data'

# # Function to process and display 10 random cards
# def process_random_cards(data_dir, num_cards=10):
#     # Get random card images
#     random_card_images = get_random_card_images(data_dir, num_cards)
    
#     for i, card_image_path in enumerate(random_card_images):
#         # Open the card image
#         image = Image.open(card_image_path).convert("RGB")
#         resized_image = image.resize(RESIZE_DIM)  # Resize for display consistency
        
#         # Display the card
#         plt.figure(figsize=(6, 6))
#         plt.imshow(resized_image)
#         plt.axis('off')
#         plt.title(f"Card {i+1}")
#         plt.show()

# # Process 10 random cards
# process_random_cards(data_dir)


# Mixed Model (Table + Discard)

In [ ]:
# Lower confidence threshold for predictions
LOW_CONFIDENCE_THRESHOLD = 0.5  # Set to your desired threshold (e.g., 0.2 for 20% confidence)

In [ ]:
# rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
# project = rf.workspace("houssem-ouerdiane-nmtbp").project("fantasy-realms")
# version = project.version(6)
# dataset = version.download("yolov5")

In [ ]:
# Define the model, specifying the pre-trained weights you want to use (e.g., yolov8n.pt)
model = YOLO("yolov8n.pt")  # Use YOLOv8 small model (yolov8n.pt)

#Train the model using the dataset specified in the data.yaml file
#model.train(data="data_mixed.yaml", epochs=150, batch=16, imgsz=640)
#already trained
#saved in train 13

In [ ]:
# Define the path to the data.yaml file
yaml_path = 'Fantasy-Realms-6/data.yaml'

# Open and load the YAML file
with open(yaml_path, 'r') as file:
    data = yaml.safe_load(file)

# Print the contents of the YAML file
print(yaml.dump(data, sort_keys=False, default_flow_style=False))

In [ ]:
# Directory containing the "Discard Data" folder
data_dir = 'Discard Data'

In [ ]:
# Roboflow model information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '6'  # The version number of your deployed model

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'

RESIZE_DIM = (640, 640)  # Resize dimensions used during model training

# Function to get 10 random image paths from the "Discard Data" folder
def get_random_images(data_dir, num_images=10):
    images = [f for f in os.listdir(data_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random_images = random.sample(images, min(num_images, len(images)))
    return [os.path.join(data_dir, img) for img in random_images]

# Function to apply predictions on a given image using the Roboflow model
def predict_image(image_path):
    # Open and resize the image
    image = Image.open(image_path).convert("RGB")
    image_resized = image.resize(RESIZE_DIM)
    
    # Save the resized image temporarily for uploading
    temp_image_path = "temp_image.jpg"
    image_resized.save(temp_image_path)
    
    # Send the image to Roboflow API for prediction
    with open(temp_image_path, "rb") as img:
        response = requests.post(
            BASE_URL,
            params={"api_key": API_KEY},
            files={"file": img}
        )
    
    # Parse the prediction results
    predictions = response.json()
    return predictions, image_resized

# Function to draw bounding boxes on the image
def draw_bounding_boxes(image, predictions):
    draw = ImageDraw.Draw(image)
    if 'predictions' in predictions and predictions['predictions']:
        for pred in predictions['predictions']:
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            confidence = pred['confidence']
            label = pred['class']
            
            # Calculate bounding box coordinates
            x_min = x - width / 2
            y_min = y - height / 2
            x_max = x + width / 2
            y_max = y + height / 2
            
            # Draw the bounding box and label
            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
            draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
    return image

# Main function to process 10 random images
def process_random_images(data_dir):
    random_images = get_random_images(data_dir)
    
    for image_path in random_images:
        predictions, resized_image = predict_image(image_path)
        image_with_boxes = draw_bounding_boxes(resized_image.copy(), predictions)
        
        # Display the image with bounding boxes
        plt.figure(figsize=(8, 8))
        plt.imshow(image_with_boxes)
        plt.axis('off')
        plt.title("Predictions with Bounding Boxes")
        plt.show()

# Apply predictions on 10 random images
process_random_images(data_dir)

In [ ]:
# Directory containing the Table Data folder
data_dir = 'Table Data'

In [ ]:
# Roboflow model information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '6'  # The version number of your deployed model

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'

RESIZE_DIM = (640, 640)  # Resize dimensions used during model training

# Function to get 10 random image paths from the directory
def get_random_images(data_dir, num_images=10):
    subfolders = [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]
    random_images = []
    
    for _ in range(num_images):
        random_folder = random.choice(subfolders)
        folder_path = os.path.join(data_dir, random_folder)
        images = [f for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
        random_image = random.choice(images)
        random_images.append(os.path.join(folder_path, random_image))
    
    return random_images

# Function to apply predictions on a given image using the Roboflow model
def predict_image(image_path):
    # Open and resize the image
    image = Image.open(image_path).convert("RGB")
    image_resized = image.resize(RESIZE_DIM)
    
    # Save the resized image temporarily for uploading
    temp_image_path = "temp_image.jpg"
    image_resized.save(temp_image_path)
    
    # Send the image to Roboflow API for prediction
    with open(temp_image_path, "rb") as img:
        response = requests.post(
            BASE_URL,
            params={"api_key": API_KEY},
            files={"file": img}
        )
    
    # Parse the prediction results
    predictions = response.json()
    return predictions, image_resized

# Function to draw bounding boxes on the image
def draw_bounding_boxes(image, predictions):
    draw = ImageDraw.Draw(image)
    if 'predictions' in predictions and predictions['predictions']:
        for pred in predictions['predictions']:
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            confidence = pred['confidence']
            label = pred['class']
            
            # Calculate bounding box coordinates
            x_min = x - width / 2
            y_min = y - height / 2
            x_max = x + width / 2
            y_max = y + height / 2
            
            # Draw the bounding box and label
            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
            draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
    return image

# Main function to process 10 random images
def process_random_images(data_dir):
    random_images = get_random_images(data_dir)
    
    for image_path in random_images:
        predictions, resized_image = predict_image(image_path)
        image_with_boxes = draw_bounding_boxes(resized_image.copy(), predictions)
        
        # Display the image with bounding boxes
        plt.figure(figsize=(8, 8))
        plt.imshow(image_with_boxes)
        plt.axis('off')
        plt.title("Predictions with Bounding Boxes")
        plt.show()


# Apply predictions on 10 random images
process_random_images(data_dir)

In [ ]:
# Load the results.csv file
results_path = 'runs/detect/train13/results.csv' #train is for hand data
df = pd.read_csv(results_path)

In [ ]:
# Display the table in a scrollable div
scrollable_table = df.to_html(index=False)
display(HTML(f"""
<div style="overflow:auto; height:400px; border:1px solid #ccc; padding:10px;">
    {scrollable_table}
</div>
"""))

In [ ]:
# Select the last epoch (max epoch)
last_epoch_data = df.loc[df['epoch'].idxmax()]

# Print each metric with its value
print("Metrics for the last epoch (Epoch {}):".format(last_epoch_data['epoch']))
for column in last_epoch_data.index:
    print(f"{column}: {last_epoch_data[column]}")

In [ ]:
# Extract epochs and precision values
epochs = df['epoch']
precision_b = df['metrics/precision(B)']

# Plot the evolution of precision over epochs
plt.figure(figsize=(10, 6))
plt.plot(epochs, precision_b, marker='o', linestyle='-', color='b')
plt.title('Evolution of Precision (B) by Epoch')
plt.xlabel('Epoch')
plt.ylabel('Precision (B)')
plt.grid(True)

# Set x-ticks at every 10 epochs
plt.xticks(ticks=epochs[::10], labels=epochs[::10], rotation=45)  

plt.tight_layout()
plt.show()

In [ ]:
# Plot 'metrics/recall(B)' by epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['metrics/recall(B)'], label='Recall (B)', color='red')
plt.xlabel('Epoch')
plt.ylabel('Recall (B)')
plt.title('Recall (B) by Epoch')
plt.xticks(df['epoch'][::10])  # Show every 10th epoch
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Plot 'metrics/mAP50(B)' by epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50 (B)', color='green')
plt.xlabel('Epoch')
plt.ylabel('mAP50 (B)')
plt.title('mAP50 (B) by Epoch')
plt.xticks(df['epoch'][::10])  # Show every 10th epoch
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Plot 'metrics/mAP50-95(B)' by epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95 (B)', color='blue')
plt.xlabel('Epoch')
plt.ylabel('mAP50-95 (B)')
plt.title('mAP50-95 (B) by Epoch')
plt.xticks(df['epoch'][::10])  # Show every 10th epoch
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Paths
train_folder="runs/detect/train13"

# List of image file names (without extensions)
image_files = [
    'confusion_matrix',
    'confusion_matrix_normalized',
    'F1_curve',
    'labels',
    'labels_correlogram',
    'P_curve',
    'PR_curve',
    'R_curve',
    'results'
]

# Check for each image file and display it if it exists
for img_name in image_files:
    for ext in ['.png', '.jpg']:
        img_path = os.path.join(train_folder, img_name + ext)
        if os.path.exists(img_path):
            image = Image.open(img_path)
            # Resize image and display
            resized_image = image.resize((1200, 800))  # Resize for better visibility
            ipd.display(resized_image)
            break

# Manuel Evaluation

In [ ]:
# Set the path to the Fresh Data directory
data_directory = "Fresh Data"

In [ ]:
# Register HEIF format with Pillow
pillow_heif.register_heif_opener()

def convert_heic_to_jpeg(directory):
    found_heic_files = 0  # Counter for HEIC files

    # Walk through the directory
    for root, _, files in os.walk(directory):
        for file in files:
            # Check if the file has a .HEIC extension
            if file.lower().endswith(".heic"):
                found_heic_files += 1  # Increment the counter
                heic_path = os.path.join(root, file)

                # Open the .HEIC image
                try:
                    image = Image.open(heic_path)
                    # Define the new .jpeg path
                    jpeg_path = os.path.splitext(heic_path)[0] + ".jpeg"
                    
                    # Save the image as .jpeg
                    image.save(jpeg_path, "JPEG")
                    
                    # Delete the original .HEIC file
                    os.remove(heic_path)
                    print(f"Converted and deleted: {heic_path}")
                except Exception as e:
                    print(f"Failed to convert {heic_path}: {e}")

    # Check if no HEIC files were found
    if found_heic_files == 0:
        print("No .HEIC images found to convert.")

# Convert all .HEIC images to .jpeg and delete the originals
convert_heic_to_jpeg(data_directory)

In [ ]:
# Path to the folder containing the table data images
table_data_path = "Fresh Data/Table Data"

# Dictionary of labels for each image
labels_dict = {
    "IMG_1789": "Blizzard",
    "IMG_1790": "Queen",
    "IMG_1791": "Hydra",
    "IMG_1792": "Sword of Keth",
    "IMG_1793": "Basilisk",
    "IMG_1794": "Doppelganger",
    "IMG_1795": "Magic Wand",
    "IMG_1796": "Earth Elemental",
    "IMG_1797": "Necromancer",
    "IMG_1798": "War Horse",
    "IMG_1799": "Gem Of Order",
    "IMG_1800": "King",
    "IMG_1801": "Rune Of Protection",
    "IMG_1802": "Beast Master",
    "IMG_1803": "Great Flood",
    "IMG_1804": "World Tree",
    "IMG_1805": "Clock Tower",
    "IMG_1806": "Shapeshifter",
    "IMG_1807": "Book Of Changes",
    "IMG_1808": "Unicorn",
    "IMG_1809": "Mountain",
    "IMG_1810": "Shield Of Keth",
    "IMG_1811": "Lightning",
    "IMG_1812": "Warlock Lord",
    "IMG_1813": "Elven Longbow",
    "IMG_1814": "Light Cavalry",
    "IMG_1815": "Island",
    "IMG_1816": "Air Elemental",
    "IMG_1817": "Dwarvish Infantry",
    "IMG_1818": "Princess",
    "IMG_1819": "Elemental Enchantress",
    "IMG_1820": "Smoke",
    "IMG_1821": "Swamp",
    "IMG_1822": "Warlord",
    "IMG_1823": "Candle",
    "IMG_1824": "Dragon",
    "IMG_1825": "Collector",
    "IMG_1826": "Rainstorm",
    "IMG_1827": "Empress",
    "IMG_1828": "Knights",
    "IMG_1829": "Underground Caverns",
    "IMG_1830": "Fountain Of Life",
    "IMG_1831": "Water Elemental",
    "IMG_1832": "Whirlwind",
    "IMG_1833": "Fire Elemental",
    "IMG_1834": "Elven Archers",
    "IMG_1835": "War Dirigable",
    "IMG_1836": "Rangers",
    "IMG_1837": "Wildfire",
    "IMG_1838": "Forge",
    "IMG_1839": "Warship",
    "IMG_1840": "Forest",
    "IMG_1841": "Mirage"
}

# Function to create the CSV
def create_label_csv(folder_path, labels, csv_folder="CSV Files", csv_name="table_data.csv"):
    # Ensure the folder to store the CSV exists
    os.makedirs(csv_folder, exist_ok=True)
    
    # Full path for the CSV file
    csv_path = os.path.join(csv_folder, csv_name)
    
    # Get the list of image files in the specified folder
    image_files = sorted(
        [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    )
    
    # Create the CSV file
    with open(csv_path, mode='w', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["image_name", "label"])  # Header
        
        for image in image_files:
            image_name, _ = os.path.splitext(image)
            label = labels.get(image_name, "Unknown")  # Default label is 'Unknown'
            writer.writerow([image, label])
    
    print(f"CSV file '{csv_name}' created successfully in folder '{csv_folder}'.")

# Create the CSV for table data
#create_label_csv(table_data_path, labels_dict) #alreadt created


In [ ]:
# Path to the CSV file
csv_path = "CSV Files/table_data.csv"

# Function to print the content of the CSV file
def print_csv_content(csv_path):
    try:
        with open(csv_path, mode='r', encoding='utf-8') as csv_file:
            reader = csv.reader(csv_file)
            for row in reader:
                print(", ".join(row))  # Print each row
    except FileNotFoundError:
        print(f"The file '{csv_path}' does not exist.")

# Print the CSV content
print_csv_content(csv_path)


In [ ]:
# Function to set the data directory
def set_data_directory(base_path="Fresh Data", subfolder="Table Data"):
    data_directory = os.path.join(base_path, subfolder)
    if not os.path.exists(data_directory):
        os.makedirs(data_directory)  # Create the directory if it doesn't exist
    return data_directory

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/fantasy-realms/6'
API_KEY = os.getenv("ROBOFLOW_API_KEY")
RESIZE_DIM = (640, 640)
CSV_PATH = os.path.join("CSV Files", "table_data.csv")

# Function to load ground truth labels from CSV
def load_ground_truth_labels(csv_path):
    labels = {}
    with open(csv_path, mode='r', encoding='utf-8') as csv_file:
        reader = csv.DictReader(csv_file)
        for row in reader:
            labels[row['image_name']] = row['label']
    return labels

# Function to get all image paths from the directory
def get_all_images(data_dir):
    return [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# Function to apply predictions on a given image using the Roboflow model
def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image_resized = image.resize(RESIZE_DIM)
    temp_image_path = "temp_image.jpg"
    image_resized.save(temp_image_path)

    with open(temp_image_path, "rb") as img:
        response = requests.post(
            BASE_URL,
            params={"api_key": API_KEY},
            files={"file": img}
        )
    predictions = response.json()
    return predictions, image_resized

# Function to draw bounding boxes on the image
def draw_bounding_boxes(image, predictions):
    draw = ImageDraw.Draw(image)
    if 'predictions' in predictions and predictions['predictions']:
        for pred in predictions['predictions']:
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            confidence = pred['confidence']
            label = pred['class']

            x_min = x - width / 2
            y_min = y - height / 2
            x_max = x + width / 2
            y_max = y + height / 2

            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
            draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
    return image

# Main function to process images and calculate precision
def process_all_images(data_dir):
    ground_truth = load_ground_truth_labels(CSV_PATH)
    all_images = get_all_images(data_dir)
    correct_predictions = 0
    total_predictions = 0
    incorrect_predictions = []

    for image_path in all_images:
        image_name = os.path.basename(image_path)
        predictions, resized_image = predict_image(image_path)

        if 'predictions' in predictions and predictions['predictions']:
            predicted_label = max(predictions['predictions'], key=lambda x: x['confidence'])['class']
            ground_truth_label = ground_truth.get(image_name, "Unknown")

            if predicted_label == ground_truth_label:
                correct_predictions += 1
            else:
                incorrect_predictions.append((image_name, predicted_label, ground_truth_label))

            total_predictions += 1
        else:
            # No predictions made for this image
            ground_truth_label = ground_truth.get(image_name, "Unknown")
            incorrect_predictions.append((image_name, "No Prediction", ground_truth_label))
            total_predictions += 1

        image_with_boxes = draw_bounding_boxes(resized_image.copy(), predictions)
        plt.figure(figsize=(8, 8))
        plt.imshow(image_with_boxes)
        plt.axis('off')
        plt.title(f"Predictions for {image_name}")
        plt.show()

    precision = (correct_predictions / total_predictions) * 100 if total_predictions > 0 else 0

    print("\nPrecision Results:")
    print(f"- Total Predictions: {total_predictions}")
    print(f"- Correct Predictions: {correct_predictions}")
    print(f"- Incorrect Predictions: {len(incorrect_predictions)}")
    print(f"- Precision Score: {precision:.2f}%")

    if incorrect_predictions:
        print("\nIncorrect Predictions:")
        for image_name, predicted_label, ground_truth_label in incorrect_predictions:
            print(f"Image: {image_name}, Predicted: {predicted_label}, Actual: {ground_truth_label}")

# Set data directory and process all images
data_dir = set_data_directory()
process_all_images(data_dir)

In [ ]:
# Function to create the CSV
def create_label_csv(folder_path, labels, csv_folder="CSV Files", csv_name="discard_data.csv"):
    # Ensure the folder to store the CSV exists
    os.makedirs(csv_folder, exist_ok=True)
    
    # Full path for the CSV file
    csv_path = os.path.join(csv_folder, csv_name)
    
    # Get the list of image files in the specified folder
    image_files = sorted(
        [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    )
    
    # Create the CSV file
    with open(csv_path, mode='w', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["image_name", "label"])  # Header
        
        for image in image_files:
            image_name, _ = os.path.splitext(image)
            label = labels.get(image_name, "Unknown")  # Default label is 'Unknown'
            writer.writerow([image, label])
    
    print(f"CSV file '{csv_name}' created successfully in folder '{csv_folder}'.")

# Data for Discard Data
discard_labels = {
    "IMG_1842": "Hydra, King",
    "IMG_1843": "Shapeshifter, Dwarvish Infantry, Rainstorm",
    "IMG_1844": "Mountain, Unicorn, Rune Of Protection, Air Elemental",
    "IMG_1845": "Knights, Book Of Changes, Elemental Enchantress, Elven Longbow, Candle",
    "IMG_1846": "War Horse, Earth Elemental, Necromancer, Wildfire, Empress, Clock Tower",
    "IMG_1847": "War Dirigable, Warlord, Mirage, Shield Of Keth, Queen, Collector, Rangers",
    "IMG_1848": "World Tree, Gem Of Order, Princess, Forest, Warship, Warlock Lord, Forge, Elven Archers",
    "IMG_1849": "Basilisk, Magic Wand, Water Elemental, Smoke, Whirlwind, Light Cavalry, Fountain Of Life, Island, Underground Caverns",
    "IMG_1850": "Lightning, Sword of Keth, Great Flood, Beast Master, Doppelganger, Dragon, Fire Elemental, Swamp, Blizzard",
    "IMG_1851": "Elemental Enchantress, Elven Longbow",
    "IMG_1852": "Empress, Wildfire, Fountain Of Life",
    "IMG_1853": "Mirage, Warlord, Dragon, Fire Elemental",
    "IMG_1854": "World Tree, Lightning, Mountain, Rainstorm, Rangers",
    "IMG_1855": "Candle, Earth Elemental, War Horse, Hydra, Light Cavalry, Beast Master",
    "IMG_1856": "Blizzard, Queen, Whirlwind, Unicorn, Collector, Forge, Shield Of Keth",
    "IMG_1857": "King, Forest, Elven Archers, Sword of Keth, Basilisk, Great Flood, Warship, Warlock Lord",
    "IMG_1858": "Doppelganger, Smoke, Necromancer, Swamp, Rune Of Protection, Air Elemental, Shapeshifter, Clock Tower, Magic Wand",
    "IMG_1859": "Gem Of Order, War Dirigable, Water Elemental, Princess, Underground Caverns, Dwarvish Infantry, Knights, Book Of Changes, Island"
}

# Path to the Discard Data images
discard_data_path = os.path.join("Fresh Data", "Discard Data")

# Name of the new CSV
discard_csv_name = "discard_data.csv"

# Create the CSV for Discard Data
create_label_csv(discard_data_path, discard_labels, csv_folder="CSV Files", csv_name=discard_csv_name)


In [ ]:
# Path to the CSV file
csv_path = "CSV Files/discard_data.csv"

# Function to print the content of the CSV file
def print_csv_content(csv_path):
    try:
        with open(csv_path, mode='r', encoding='utf-8') as csv_file:
            reader = csv.reader(csv_file)
            for row in reader:
                print(", ".join(row))  # Print each row
    except FileNotFoundError:
        print(f"The file '{csv_path}' does not exist.")

# Print the CSV content
print_csv_content(csv_path)

In [ ]:
# Function to set the data directory
def set_data_directory(base_path="Fresh Data", subfolder="Discard Data"):
    data_directory = os.path.join(base_path, subfolder)
    if not os.path.exists(data_directory):
        os.makedirs(data_directory)  # Create the directory if it doesn't exist
    return data_directory

# Example usage
data_dir = set_data_directory()

# Roboflow model information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '6'  # The version number of your deployed model

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'

RESIZE_DIM = (640, 640)  # Resize dimensions used during model training

# Function to get all image paths from the "Discard Data" folder
def get_all_images(data_dir):
    return [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# Function to apply predictions on a given image using the Roboflow model
def predict_image(image_path):
    # Open and resize the image
    image = Image.open(image_path).convert("RGB")
    image_resized = image.resize(RESIZE_DIM)

    # Save the resized image temporarily for uploading
    temp_image_path = "temp_image.jpg"
    image_resized.save(temp_image_path)

    # Send the image to Roboflow API for prediction
    with open(temp_image_path, "rb") as img:
        response = requests.post(
            BASE_URL,
            params={"api_key": API_KEY},
            files={"file": img}
        )

    # Parse the prediction results
    predictions = response.json()
    return predictions, image_resized

# Function to draw bounding boxes on the image
def draw_bounding_boxes(image, predictions):
    draw = ImageDraw.Draw(image)
    if 'predictions' in predictions and predictions['predictions']:
        for pred in predictions['predictions']:
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            confidence = pred['confidence']
            label = pred['class']

            # Calculate bounding box coordinates
            x_min = x - width / 2
            y_min = y - height / 2
            x_max = x + width / 2
            y_max = y + height / 2

            # Draw the bounding box and label
            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
            draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
    return image

# Function to load labels from the CSV file
def load_csv_labels(csv_folder="CSV Files", csv_name="discard_data.csv"):
    csv_path = os.path.join(csv_folder, csv_name)
    labels_dict = {}
    with open(csv_path, mode='r', encoding='utf-8') as csv_file:
        reader = csv.DictReader(csv_file)
        for row in reader:
            labels_dict[row['image_name']] = set(row['label'].split(', '))
    return labels_dict

# Function to compare predictions with the CSV labels
def compare_predictions_with_csv(data_dir, csv_labels):
    all_images = get_all_images(data_dir)
    total_predictions = 0
    total_correct = 0
    incorrect_predictions = []

    for image_path in all_images:
        image_name = os.path.basename(image_path)
        csv_labels_for_image = csv_labels.get(image_name, set())

        predictions, resized_image = predict_image(image_path)
        predicted_labels = {pred['class'] for pred in predictions.get('predictions', [])}
        total_predictions += len(predicted_labels)

        # Identify mistakes
        extra_labels = predicted_labels - csv_labels_for_image
        missing_labels = csv_labels_for_image - predicted_labels

        # Calculate the number of correct predictions
        correct_predictions = predicted_labels & csv_labels_for_image
        total_correct += len(correct_predictions)

        # Collect incorrect predictions for reporting
        if extra_labels or missing_labels:
            incorrect_predictions.append({
                "image": image_name,
                "extra": ", ".join(extra_labels),
                "missing": ", ".join(missing_labels)
            })

        # Draw and display the image with bounding boxes
        image_with_boxes = draw_bounding_boxes(resized_image.copy(), predictions)
        plt.figure(figsize=(8, 8))
        plt.imshow(image_with_boxes)
        plt.axis('off')
        plt.title(f"Predictions for {image_name}")
        plt.show()

    # Calculate precision
    precision = total_correct / total_predictions if total_predictions > 0 else 0

    # Print results
    print("========== Precision Results ==========")
    print(f"- Total Predictions: {total_predictions}")
    print(f"- Correct Predictions: {total_correct}")
    print(f"- Incorrect Predictions: {len(incorrect_predictions)}")
    print(f"- Precision Score: {precision * 100:.2f}%\n")

    if incorrect_predictions:
        print("Incorrect Predictions:")
        for error in incorrect_predictions:
            print(f"Image: {error['image']}")
            if error['extra']:
                print(f"  Extra Labels: {error['extra']}")
            if error['missing']:
                print(f"  Missing Labels: {error['missing']}\n")
    else:
        print("No incorrect predictions.")
    print("=======================================")

# Load the labels from the CSV
csv_labels = load_csv_labels()

# Compare predictions with the CSV labels
compare_predictions_with_csv(data_dir, csv_labels)



In [ ]:
# Define the labels for the Hand data
hand_data_labels = {
    "1860": "War Dirigible, Island, Doppelganger, Smoke, Basilisk, Magic Wand, Princess",
    "1861": "Candle, Sword of Keth, Dwarvish Infantry, Underground Caverns, Great Flood, Forge, Forest",
    "1862": "Shield of Keth, Fountain of Life, Knights, Wildfire, Swamp, Empress, King",
    "1863": "Lightning, Air Elemental, Book of Changes, Rune of Protection, Elemental Enchantress, Unicorn, Rainstorm",
    "1864": "Elven Archers, Shapeshifter, Light Cavalry, Gem of Order, Beast Master, Warlord, Hydra",
    "1865": "Warship, Water Elemental, World Tree, Necromancer, Clock Tower, Collector, Earth Elemental",
    "1866": "Whirlwind, War Horse, Warlock Lord, Mountain, Dragon, Rangers, Queen",
    "1867": "Fire Elemental, Mirage, Blizzard, Elven Longbow, Rainstorm, Empress, Smoke"
}

# Function to create the CSV
def create_hand_data_csv(folder_path, labels, csv_folder="CSV Files", csv_name="hand_data.csv"):
    # Ensure the folder to store the CSV exists
    os.makedirs(csv_folder, exist_ok=True)
    
    # Full path for the CSV file
    csv_path = os.path.join(csv_folder, csv_name)
    
    # Get the list of image files in the specified folder
    image_files = sorted(
        [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    )
    
    # Create the CSV file
    with open(csv_path, mode='w', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["image_name", "label"])  # Header
        
        for image in image_files:
            image_name, _ = os.path.splitext(image)
            label = labels.get(image_name.split('_')[1], "Unknown")  # Match labels using the image ID
            writer.writerow([image, label])
    
    print(f"CSV file '{csv_name}' created successfully in folder '{csv_folder}'.")

# Define the folder path for Hand Data
hand_data_folder = "Fresh Data/Hand Data"

# Create the CSV for Hand Data
create_hand_data_csv(hand_data_folder, hand_data_labels)



In [ ]:
# Path to the CSV file
csv_path = "CSV Files/hand_data.csv"

# Function to print the content of the CSV file
def print_csv_content(csv_path):
    try:
        with open(csv_path, mode='r', encoding='utf-8') as csv_file:
            reader = csv.reader(csv_file)
            for row in reader:
                print(", ".join(row))  # Print each row
    except FileNotFoundError:
        print(f"The file '{csv_path}' does not exist.")

# Print the CSV content
print_csv_content(csv_path)

In [ ]:
import os
import csv
import requests
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Function to set the data directory
def set_data_directory(base_path="Fresh Data", subfolder="Hand Data"):
    data_directory = os.path.join(base_path, subfolder)
    if not os.path.exists(data_directory):
        os.makedirs(data_directory)  # Create the directory if it doesn't exist
    return data_directory

# Example usage
data_dir = set_data_directory()

# Roboflow model information
API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
VERSION_NUMBER = '5'  # The version number of your deployed model

# Base URL for Roboflow API requests
BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'

RESIZE_DIM = (640, 640)  # Resize dimensions used during model training
CONFIDENCE_THRESHOLD = 0.1  # Set confidence threshold to 0.1

# Function to get all image paths from the "Hand Data" folder
def get_all_images(data_dir):
    return [os.path.join(data_dir, img) for img in os.listdir(data_dir) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]

# Function to apply predictions on a given image using the Roboflow model
def predict_image(image_path):
    # Open and resize the image
    image = Image.open(image_path).convert("RGB")
    image_resized = image.resize(RESIZE_DIM)
    
    # Save the resized image temporarily for uploading
    temp_image_path = "temp_image.jpg"
    image_resized.save(temp_image_path)
    
    # Send the image to Roboflow API for prediction
    with open(temp_image_path, "rb") as img:
        response = requests.post(
            BASE_URL,
            params={
                "api_key": API_KEY,
                "confidence": CONFIDENCE_THRESHOLD  # Add confidence parameter here
            },
            files={"file": img}
        )
    
    # Parse the prediction results
    predictions = response.json()
    return predictions, image_resized

# Function to draw bounding boxes on the image
def draw_bounding_boxes(image, predictions):
    draw = ImageDraw.Draw(image)
    prediction_text = []

    if 'predictions' in predictions and predictions['predictions']:
        # Sort predictions by confidence in descending order
        sorted_predictions = sorted(predictions['predictions'], key=lambda x: x['confidence'], reverse=True)
        
        # Keep only the top 7 predictions
        top_predictions = sorted_predictions[:7]

        for pred in top_predictions:
            x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
            confidence = pred['confidence']
            label = pred['class']
            
            # Calculate bounding box coordinates
            x_min = x - width / 2
            y_min = y - height / 2
            x_max = x + width / 2
            y_max = y + height / 2
            
            # Draw the bounding box and label
            draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
            draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
            
            # Append prediction details for display
            prediction_text.append(f"{label}: {confidence:.2f}")
    return image, prediction_text

# Function to process all images and calculate precision
def process_all_images(data_dir, csv_path="CSV Files/hand_data.csv"):
    # Load ground truth labels
    ground_truth = {}
    with open(csv_path, mode='r', encoding='utf-8') as csv_file:
        reader = csv.DictReader(csv_file)
        for row in reader:
            ground_truth[row['image_name']] = {label.lower() for label in row['label'].split(', ')}

    all_images = get_all_images(data_dir)
    total_predictions = 0
    total_correct = 0
    incorrect_predictions = []

    for image_path in all_images:
        image_name = os.path.basename(image_path)
        ground_truth_labels = ground_truth.get(image_name, set())
        
        predictions, resized_image = predict_image(image_path)

        # Sort predictions by confidence and take the top 7
        sorted_predictions = sorted(predictions.get('predictions', []), key=lambda x: x['confidence'], reverse=True)[:7]
        predicted_labels = {pred['class'].lower() for pred in sorted_predictions}
        
        total_predictions += len(predicted_labels)
        correct_labels = predicted_labels & ground_truth_labels
        incorrect_labels = (predicted_labels - ground_truth_labels) | (ground_truth_labels - predicted_labels)
        total_correct += len(correct_labels)

        if incorrect_labels:
            incorrect_predictions.append((image_name, incorrect_labels))

        # Draw bounding boxes on the image
        image_with_boxes, _ = draw_bounding_boxes(resized_image.copy(), {"predictions": sorted_predictions})
        plt.figure(figsize=(8, 8))
        plt.imshow(image_with_boxes)
        plt.axis('off')
        plt.title(f"Predictions for {image_name}")
        plt.show()

    precision = (total_correct / total_predictions * 100) if total_predictions > 0 else 0

    print("\nPrecision Results:")
    print(f"- Total Predictions: {total_predictions}")
    print(f"- Correct Predictions: {total_correct}")
    print(f"- Incorrect Predictions: {len(incorrect_predictions)}")
    print(f"- Precision Score: {precision:.2f}%")

    if incorrect_predictions:
        print("\nIncorrect Predictions:")
        for image_name, errors in incorrect_predictions:
            print(f"Image: {image_name}, Incorrect Labels: {errors}")

# Example usage
process_all_images(data_dir)


In [ ]:
# the above code is still buggy

In [ ]:
# def set_data_directory(base_path="Fresh Data", subfolder="Hand Data"):
#     data_directory = os.path.join(base_path, subfolder)
#     if not os.path.exists(data_directory):
#         os.makedirs(data_directory)  # Create the directory if it doesn't exist
#     return data_directory

# # Example usage
# data_dir = set_data_directory()

# # Roboflow model information
# API_KEY = os.getenv("ROBOFLOW_API_KEY")  # Your Roboflow API key
# MODEL_NAME = 'fantasy-realms'  # The model name from Roboflow
# VERSION_NUMBER = '5'  # The version number of your deployed model

# # Base URL for Roboflow API requests
# BASE_URL = f'https://detect.roboflow.com/{MODEL_NAME}/{VERSION_NUMBER}'

# RESIZE_DIM = (640, 640)  # Resize dimensions used during model training
# CONFIDENCE_THRESHOLD = 0.1  # Set confidence threshold to 0.1

# # Function to get 10 random image paths from the "Discard Data" folder
# def get_random_images(data_dir, num_images=10):
#     images = [f for f in os.listdir(data_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
#     random_images = random.sample(images, min(num_images, len(images)))
#     return [os.path.join(data_dir, img) for img in random_images]

# # Function to apply predictions on a given image using the Roboflow model
# def predict_image(image_path):
#     # Open and resize the image
#     image = Image.open(image_path).convert("RGB")
#     image_resized = image.resize(RESIZE_DIM)
    
#     # Save the resized image temporarily for uploading
#     temp_image_path = "temp_image.jpg"
#     image_resized.save(temp_image_path)
    
#     # Send the image to Roboflow API for prediction
#     with open(temp_image_path, "rb") as img:
#         response = requests.post(
#             BASE_URL,
#             params={
#                 "api_key": API_KEY,
#                 "confidence": CONFIDENCE_THRESHOLD  # Add confidence parameter here
#             },
#             files={"file": img}
#         )
    
#     # Parse the prediction results
#     predictions = response.json()
#     return predictions, image_resized

# # Function to draw bounding boxes on the image
# def draw_bounding_boxes(image, predictions):
#     draw = ImageDraw.Draw(image)
#     prediction_text = []

#     if 'predictions' in predictions and predictions['predictions']:
#         # Sort predictions by confidence in descending order
#         sorted_predictions = sorted(predictions['predictions'], key=lambda x: x['confidence'], reverse=True)
        
#         # Keep only the top 7 predictions
#         top_predictions = sorted_predictions[:7]

#         for pred in top_predictions:
#             x, y, width, height = pred['x'], pred['y'], pred['width'], pred['height']
#             confidence = pred['confidence']
#             label = pred['class']
            
#             # Calculate bounding box coordinates
#             x_min = x - width / 2
#             y_min = y - height / 2
#             x_max = x + width / 2
#             y_max = y + height / 2
            
#             # Draw the bounding box and label
#             draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
#             draw.text((x_min, y_min), f"{label} ({confidence:.2f})", fill="red")
            
#             # Append prediction details for display
#             prediction_text.append(f"{label}: {confidence:.2f}")
#     return image, prediction_text

# # Main function to process 10 random images
# def process_random_images(data_dir):
#     random_images = get_random_images(data_dir)
    
#     for image_path in random_images:
#         predictions, resized_image = predict_image(image_path)
#         image_with_boxes, prediction_text = draw_bounding_boxes(resized_image.copy(), predictions)
        
#         # Display the image with bounding boxes
#         plt.figure(figsize=(8, 8))
#         plt.imshow(image_with_boxes)
#         plt.axis('off')
#         plt.title("Predictions with Bounding Boxes")
#         plt.show()
        
#         # Display predictions below the image
#         display(Markdown("### Predictions:"))
#         for text in prediction_text:
#             display(Markdown(f"- {text}"))

# # Example usage
# process_random_images(data_dir)

In [ ]:
# notes
# calculate precision for fresh data
# include the previous in the reportss reports